In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter, NullFormatter, ScalarFormatter
import pandas as pd
import seaborn as sb
import seaborn.objects as so
import sys
from matplotlib.offsetbox import AnchoredText


In [ ]:

df = pd.read_csv("results/NeoN-GPU.csv", skip_blank_lines=True)
df["time_ms"] = df["time_us"] / 1000
df["ms_normed"] = df["time_ms"] / df["cells"]
df["us_normed"] = df["time_us"] / df["cells"]
df["mean_normed"] = df.groupby(["cells", "julia_or_neon", "strategy"])[
        "ms_normed"
].transform("median")
df["us_mean_normed"] = df.groupby(["cells", "julia_or_neon", "strategy"])[
        "us_normed"
].transform("median")
prefix = "gpustrats"
# df = df[df["strategy"] == "Cell-Based"]
cells = sorted(df[~df["cells"].isna()]["cells"].unique())
nnz = sorted(df[~df["nnz"].isna()]["nnz"].unique())
print(nnz)
df["nnz"] = df["nnz"].fillna(df["cells"].map(lambda c: nnz[cells.index(c)]))
print(df["nnz"].unique())
df["merged"] = df["strategy"] + df["julia_or_neon"]
df["merged"] = df["merged"].apply(lambda s: s.replace("JuNe", " (JuNe)").replace("NeoN", " (NeoN)"))
df["bandwidth"] = df["nnz"] * 24  / (df["time_ms"] * 1000000) 
print(df[(df["node"] == "gpu-nvidia-h200") & (df["julia_or_neon"] == "NeoN + Julia")]["cells"].unique())


In [ ]:

def time_normed(_df, executor):
    sb.set_theme(rc={'figure.figsize':(10, 5)})
    sb.set_style("whitegrid")
    sb.color_palette("deep", 8)
    with sb.plotting_context("paper", font_scale=1.7):
        p = sb.lineplot(
            data=_df,
            x="cells",
            y="mean_normed",
            hue="merged",
            style="merged",
            markers=True,
            markersize=12,
            dashes=False
        )

        p.legend(title=executor)
        p.set(
            ylabel="Assembly Time per Cell [ms]",
            xlabel="#Cells",
            yscale="log",
            xscale="log",
        )
        plt.tight_layout()
        print(f"writing into {prefix}_{executor}_normed.svg")
    plt.savefig(f"{prefix}_{executor}_normed.svg")
    plt.clf()
    plt.cla()
    plt.close()


In [ ]:


def time(_df, executor):
    sb.set_theme(rc={'figure.figsize':(10, 5)})
    sb.set_style("whitegrid")
    sb.color_palette("deep", 8)
    with sb.plotting_context("paper", font_scale=1.7):
        p = sb.lineplot(
            data=_df,
            x="cells",
            y="time_ms",
            hue="julia_or_neon",
            style="julia_or_neon",
            markers=True,
            markersize=12,
            dashes=False
        )

        p.legend(title=executor)
        p.set(
            ylabel="Assembly Time per Cell [ms]",
            xlabel="#Cells",
            yscale="log",
            xscale="log",
        )
        plt.tight_layout()
        print(f"writing into {prefix}_{executor}.svg")
    plt.savefig(f"{prefix}_{executor}.svg")
    plt.clf()
    plt.cla()
    plt.close()


In [ ]:

def time_by_threads(_df, executor):
    group_cols = [
        "case",
        "strategy",
        "julia_or_neon",
    ]
    baseline = (
        _df[_df["threads"] == 1]
        .groupby(group_cols, as_index=False)["time_ms"]
        .first()
        .rename(columns={"time_ms": "serial_time"})
    )
    _df = _df.merge(baseline, on=group_cols, how="left")
    _df["speedup"] = _df["serial_time"] / _df["time_ms"]
    _df["efficiency"] = _df["speedup"] / _df["threads"]

    sb.set_theme(rc={'figure.figsize':(10, 5)})
    sb.set_style("whitegrid")
    sb.color_palette("deep", 8)
    ts = _df["threads"].unique()
    with sb.plotting_context("paper", font_scale=1.7):
        
        p = sb.lineplot(
            data=_df,
            x="threads",
            y="speedup",
            hue="julia_or_neon",
            style="julia_or_neon",
            markers=True,
            markersize=12,
            dashes=False
        )
        ax = p.axes
        # ax.plot([1, 128],[1,128], label="Ideal Speedup")
        p.legend(title=executor)
        p.set(
            ylabel="Speedup",
            xlabel="#Threads",
            xscale="log",
            xticks=ts,
            xticklabels=[str(t) for t in ts],
        )
        plt.tight_layout()
        print(f"writing into {prefix}_{executor}_speedup.svg")
        # handles, _ = ax.get_legend_handles_labels()
        # ax.legend_.remove()

        # ax.legend(
        #     handles,
        #     _df["julia_or_neon"].unique(),
        #     loc="upper left",
        #     title="Strategy"
        # )
    plt.savefig(f"{prefix}_{executor}_speedup.svg")
    plt.clf()
    plt.cla()
    plt.close()
executors = df["executor"].unique()
print(f"executors : {executors}")


In [ ]:

# def strategies(_df, executor):
#     sb.set_theme(rc={'figure.figsize':(10, 5)})
#     sb.set_style("whitegrid")
#     sb.color_palette("deep", 8)
#     nodes = _df["node"].unique()
#     with sb.plotting_context("paper", font_scale=1.7):
#         fig, axes = plt.subplots(1, len(nodes))
#         for ax, node in zip(axes, nodes):
#             p = sb.lineplot(
#                 data=_df[_df["node"] == node],
#                 x="cells",
#                 y="ms_normed",
#                 hue="merged",
#                 err_style=None,
#                 style="merged",
#                 markers=True,
#                 markersize=12,
#                 dashes=False,
#                 ax=ax,
#             )

#             p.legend(title=node)
#             p.set(
#                 ylabel="Assembly Time per Cell [ms]",
#                 xlabel="#Cells",
#                 yscale="log",
#                 xscale="log",
#             )
#         handles, labels = axes[0].get_legend_handles_labels()
#         leg = axes[0].get_legend()
#         if leg is not None:
#             leg.remove()
#         fig.legend(
#             handles,
#             labels,
#             loc="center right",
#             bbox_to_anchor=(1, 0.5),
#             ncol=1,
#             # title="Fused Strategy",
#             frameon=False,
#         )
#         fig.subplots_adjust(
#             left=0.05,
#             right=0.99,
#             # bottom=0.18,
#             # top=0.78,
#             wspace=0.25,
#         )
#         print(f"writing into {prefix}_{executor}_all.svg")
#     plt.savefig(f"{prefix}_{executor}_all.svg")
#     plt.clf()
#     plt.cla()
#     plt.close()

    

# for executor in executors:
#     if "CPU-Parallel" == executor:
#         time_by_threads(df[(df["executor"] == executor)], executor)
#     time_normed(df[df["executor"] == executor], executor)
#     time(df[df["executor"] == executor], executor)
# time(df[df["executor"] == "GPU"], "GPU")
# strategies(df, "GPU")
# df.to_csv("interface_strats.csv")


In [ ]:

sb.set_theme(rc={'figure.figsize':(20, 12)})
sb.set_style("whitegrid")
sb.color_palette("deep", 8)
nodes = df["node"].unique()
print(df["node"].unique())
for node in nodes:
    with sb.plotting_context("paper", font_scale=1.7):
        plot_df = df[df["node"] == node].copy()
        p = sb.relplot(
            data=plot_df,
            x="cells",
            y="bandwidth",
            style="merged",
            err_style=None,
            hue="merged",
            kind="line",
            markers=True,
            markersize=12
        )
        p.set(
            # yscale="log",
            xscale="log",
            xlabel="#Cells",
            ylabel="Matrix Assembly Bandwidth (GB/s)"
        )
        legend_heights = plot_df.groupby(["merged", "cells"], as_index=False)["bandwidth"].mean()
        rightmost_points = legend_heights.loc[legend_heights.groupby("merged")["cells"].idxmax()]
        legend_order = rightmost_points.sort_values("bandwidth", ascending=False)["merged"].tolist()

        handles, labels = p.axes.flat[0].get_legend_handles_labels()
        if p._legend is not None:
            p._legend.remove()
            p._legend = None
        print(f"labels: {labels}")
        by_label = dict(zip(labels, handles))

        ordered_labels = [label for label in legend_order if label in by_label]
        ordered_handles = [by_label[label] for label in ordered_labels]
        name = "NVIDIA H100" if "h100" in node else "NVIDIA H200" 
        # at = AnchoredText(
        #     "NVIDIA H100",
        #     loc="upper left",
        #     # bbox_to_anchor=(0.975, 0.92),
        #     bbox_transform=p.axes.flat[0].transAxes,
        #     frameon=True,
        #     prop={"size": plt.rcParams["legend.fontsize"]},
        # )
        # p.axes.flat[0].add_artist(at)  
        at = AnchoredText(
            name,
            loc="upper left",
            # bbox_to_anchor=(0.975, 0.92),
            bbox_transform=p.axes.flat[0].transAxes,
            frameon=True,
            prop={"size": plt.rcParams["legend.fontsize"]},
        )
        p.axes.flat[0].add_artist(at)    
        p.set_titles("")

        p.fig.legend(
            ordered_handles,
            ordered_labels,
            loc="center left",
            bbox_to_anchor=(0.73, 0.5),
            ncol=1,
            labelspacing=1.6,
            # title="Fused Strategy",
            frameon=False,
        )
        p.fig.subplots_adjust(
            left=0.07,
            right=0.70,
            # bottom=0.18,
            top=0.92,
            # wspace=0.1,
        )
        plt.savefig(f"gpu_interface_strategies_{node}.svg", bbox_inches='tight')



In [ ]:

sb.set_theme(rc={'figure.figsize':(20, 12)})
sb.set_style("whitegrid")
sb.color_palette("deep", 8)
nodes = df["node"].unique()
print(df["node"].unique())
with sb.plotting_context("paper", font_scale=1.7):
    # plot_df = df[df["node"] == node].copy()
    p = sb.relplot(
        data=df,
        x="cells",
        y="bandwidth",
        style="merged",
        col="node",
        col_order=["gpu-nvidia-h100", "gpu-nvidia-h200"],
        err_style=None,
        hue="merged",
        kind="line",
        markers=True,
        markersize=12
    )
    p.set(
        # yscale="log",
        xscale="log",
        xlabel="#Cells",
        ylabel="Matrix Assembly Bandwidth (GB/s)"
    )
    legend_heights = plot_df.groupby(["merged", "cells"], as_index=False)["bandwidth"].mean()
    rightmost_points = legend_heights.loc[legend_heights.groupby("merged")["cells"].idxmax()]
    legend_order = rightmost_points.sort_values("bandwidth", ascending=False)["merged"].tolist()

    handles, labels = p.axes.flat[0].get_legend_handles_labels()
    if p._legend is not None:
        p._legend.remove()
        p._legend = None
    print(f"labels: {labels}")
    by_label = dict(zip(labels, handles))

    ordered_labels = [label for label in legend_order if label in by_label]
    ordered_handles = [by_label[label] for label in ordered_labels]
    at = AnchoredText(
        "NVIDIA H100",
        loc="upper left",
        # bbox_to_anchor=(0.975, 0.92),
        bbox_transform=p.axes.flat[0].transAxes,
        frameon=True,
        prop={"size": plt.rcParams["legend.fontsize"]},
    )
    p.axes.flat[0].add_artist(at)  
    at = AnchoredText(
        "NVIDIA H200",
        loc="upper left",
        # bbox_to_anchor=(0.975, 0.92),
        bbox_transform=p.axes.flat[1].transAxes,
        frameon=True,
        prop={"size": plt.rcParams["legend.fontsize"]},
    )
    p.axes.flat[1].add_artist(at)    
    p.set_titles("")
    handles, labels = p.axes.flat[0].get_legend_handles_labels()
    for ax in p.axes.flat:
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

    # # One shared legend above all subplots, 4 columns
    p.fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.0),
        ncol=4,
        # title="Fused Strategy",
        frameon=False,
    )
    # plt.tight_layout()
    p.fig.subplots_adjust(
        left=0.07,
        right=0.98,
        # bottom=0.18,
        top=0.84,
        wspace=0.05,
    )
    plt.savefig(f"gpu_interface_strategies_both.svg", bbox_inches='tight')

